## 1. Loading participant data

In [1]:
import os
# Store the current directory
current_directory = os.getcwd()
# Change to the parent directory only if not already in the parent
if os.path.basename(current_directory) != os.path.basename(os.path.abspath(os.path.join(current_directory, os.pardir))):
    os.chdir(os.pardir)

In [2]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Union


import random


class ParticipantDataLoader:
    """
    A class to load and manage participant data from the wine_quality.csv file.
    Allows easy access to trials for specific participants and phases.
    """
    
    def __init__(self, csv_path: str):
        """
        Initialize the data loader with the CSV file.
        
        Args:
            csv_path (str): Path to the wine_quality.csv file
        """
        self.data = pd.read_csv(csv_path)
        self.participants = self._get_participant_info()
        
    def _get_participant_info(self) -> Dict:
        """Get unique participant information."""
        participant_info = {}
        for participant_id in self.data['Participant Id'].unique():
            participant_data = self.data[self.data['Participant Id'] == participant_id].iloc[0]
            participant_info[participant_id] = {
                'condition': participant_data['Condition'],
                'model': participant_data['Model'],
                'app_id': participant_data['AppId'],
                'complexity': participant_data['Complexity']
            }
        return participant_info
    
    def get_participant_ids(self) -> List:
        """Get list of all participant IDs."""
        return list(self.participants.keys())
    
    def get_participant_info(self, participant_id) -> Dict:
        """Get general info for a specific participant."""
        return self.participants.get(participant_id, {})
    
    def get_participant_trials(self, participant_id, phase: Optional[str] = None) -> pd.DataFrame:
        """
        Get all trials for a specific participant.
        
        Args:
            participant_id: The participant ID
            phase (str, optional): Filter by phase ('forward' or 'counterfactual')
            
        Returns:
            pd.DataFrame: Filtered trial data
        """
        participant_data = self.data[self.data['Participant Id'] == participant_id]
        
        if phase:
            participant_data = participant_data[participant_data['Phase'] == phase]
            
        return participant_data.sort_values('Trial Index')
    
    def get_forward_trials(self, participant_id) -> pd.DataFrame:
        """
        Get forward phase trials for a participant with relevant columns.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            pd.DataFrame: Forward trial data with relevant columns
        """
        forward_data = self.get_participant_trials(participant_id, 'forward')
        
        forward_columns = [
            'Participant Id', 'Trial Index', 'Instance Id', 'XAIType', 'Tested w/ XAI', 'Time',
            'Response', 'AI prediction', 'DT prediction', 'LR prediction', 
            'Explainer prediction', 'Response==AI', 'Response==DT', 
            'Response==LR', 'Response==Explainer'
        ]
        
        # Only include columns that exist in the data
        available_columns = [col for col in forward_columns if col in forward_data.columns]
        
        return forward_data[available_columns]
    
    def get_counterfactual_trials(self, participant_id) -> pd.DataFrame:
        """
        Get counterfactual phase trials for a participant with relevant columns.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            pd.DataFrame: Counterfactual trial data with relevant columns
        """
        cf_data = self.get_participant_trials(participant_id, 'counterfactual')
        
        cf_columns = [
            'Participant Id', 'Trial Index', 'Instance Id', 'XAIType', 'Tested w/ XAI', 'Time',
            'Changed feature index', 'Changed feature name', 'Changed feature type',
            'Changed from', 'Changed to', 'Changed amount', 'DT prediction (CF)',
            'LR prediction (CF)', 'Changed prediction (DT)', 'Changed prediction (LR)',
            'Changed explainer'
        ]
        
        # Only include columns that exist in the data
        available_columns = [col for col in cf_columns if col in cf_data.columns]
        
        return cf_data[available_columns]
    
    def get_trial_by_index(self, participant_id, trial_index: int) -> pd.DataFrame:
        """
        Get a specific trial by trial index for a participant.
        
        Args:
            participant_id: The participant ID
            trial_index (int): The trial index number
            
        Returns:
            pd.DataFrame: Single trial data
        """
        participant_data = self.get_participant_trials(participant_id)
        return participant_data[participant_data['Trial Index'] == trial_index]
    
    def get_xai_performance(self, participant_id, xai_type: str = None) -> pd.DataFrame:
        """
        Get trials filtered by XAI type and whether XAI was shown.
        
        Args:
            participant_id: The participant ID
            xai_type (str, optional): Filter by specific XAI type
            
        Returns:
            pd.DataFrame: Filtered trial data
        """
        participant_data = self.get_participant_trials(participant_id)
        
        if xai_type:
            participant_data = participant_data[participant_data['XAIType'] == xai_type]
            
        return participant_data
    
    def summarize_participant_performance(self, participant_id) -> Dict:
        """
        Get a summary of participant performance across both phases.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            Dict: Summary statistics
        """
        forward_trials = self.get_forward_trials(participant_id)
        cf_trials = self.get_counterfactual_trials(participant_id)
        
        summary = {
            'participant_info': self.get_participant_info(participant_id),
            'total_trials': len(self.get_participant_trials(participant_id)),
            'forward_trials': len(forward_trials),
            'counterfactual_trials': len(cf_trials),
            'avg_time_forward': forward_trials['Time'].mean() if not forward_trials.empty else 0,
            'avg_time_counterfactual': cf_trials['Time'].mean() if not cf_trials.empty else 0,
        }
        
        # Add accuracy metrics for forward trials if available
        if not forward_trials.empty:
            accuracy_columns = ['Response==AI', 'Response==DT', 'Response==LR', 'Response==Explainer']
            for col in accuracy_columns:
                if col in forward_trials.columns:
                    summary[f'accuracy_{col.split("==")[1].lower()}'] = forward_trials[col].mean()
        
        return summary
    
    def get_instance_data(self, instance_id: int) -> pd.DataFrame:
        """
        Get all trials for a specific instance across all participants.
        
        Args:
            instance_id (int): The instance ID
            
        Returns:
            pd.DataFrame: All trials for the instance
        """
        return self.data[self.data['Instance Id'] == instance_id]
    
    def compare_participants(self, participant_ids: List) -> pd.DataFrame:
        """
        Compare performance metrics across multiple participants.
        
        Args:
            participant_ids (List): List of participant IDs to compare
            
        Returns:
            pd.DataFrame: Comparison metrics
        """
        comparison_data = []
        
        for pid in participant_ids:
            summary = self.summarize_participant_performance(pid)
            comparison_data.append({
                'Participant Id': pid,
                **summary['participant_info'],
                'Total Trials': summary['total_trials'],
                'Forward Trials': summary['forward_trials'],
                'Counterfactual Trials': summary['counterfactual_trials'],
                'Avg Time Forward': summary['avg_time_forward'],
                'Avg Time Counterfactual': summary['avg_time_counterfactual'],
                **{k: v for k, v in summary.items() if k.startswith('accuracy_')}
            })
        
        return pd.DataFrame(comparison_data)

# Example usage:
# loader = ParticipantDataLoader('datasets/wine_quality.csv')
# participant_ids = loader.get_participant_ids()
# print(f"Found {len(participant_ids)} participants")
# 
# # Get data for first participant
# first_participant = participant_ids[0]
# forward_trials = loader.get_forward_trials(first_participant)
# cf_trials = loader.get_counterfactual_trials(first_participant)
# summary = loader.summarize_participant_performance(first_participant)
# 
# print(f"Participant {first_participant} summary:")
# print(summary)

In [3]:

# Example usage:
loader = ParticipantDataLoader('datasets/combined.csv')
participant_ids = loader.get_participant_ids()
forward_trials = loader.get_forward_trials(participant_ids[0])
cf_trials = loader.get_counterfactual_trials(participant_ids[0])
summary = loader.summarize_participant_performance(participant_ids[0])

## 2. Load the AI dataset loader, explainers and forward simulation methods

In [4]:
from src.utils import AIDatasetLoader, filter_by_app_and_model, DecisionTreeInterpreter, LogisticRegressionInterpreter  
from src.memory import Chunk, DeclarativeMemory

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# # 🔧 Choose dataset here:
# app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# # 🧠 Auto-configured values:
# model_name = dataset_model_map[app_id]

In [6]:
# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

In [7]:
# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
# dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=2)
# dt_exp.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
# lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant="sparse")

In [8]:
# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "mlp",
    "adult": "mlp",
}

# 🔧 Choose dataset here:
# 📋 Dataset mapping dictionary
dataset_mapping = {
    1: "wine_quality",
    2: "mushrooms", 
    3: "forest_cover",
    4: "adult"
}

app_id1 = dataset_mapping[1]  # wine_quality
app_id2 = dataset_mapping[2]  # mushrooms
app_id3 = dataset_mapping[3]  # forest_cover
app_id4 = dataset_mapping[4]  # adult

# 🧠 Auto-configured values:
model_name1 = dataset_model_map[app_id1]
model_name2 = dataset_model_map[app_id2]
model_name3 = dataset_model_map[app_id3]
model_name4 = dataset_model_map[app_id4]

# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

ai_data_loader1 = filter_by_app_and_model(ai_dataset_loader, app_id1, model_name1)
ai_data_loader2 = filter_by_app_and_model(ai_dataset_loader, app_id2, model_name2)
ai_data_loader3 = filter_by_app_and_model(ai_dataset_loader, app_id3, model_name3)
ai_data_loader4 = filter_by_app_and_model(ai_dataset_loader, app_id4, model_name4)

# instances, preds = ai_dataset_loader.load_instances(list(range(20)), normalize=False)  # Load instances

# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
dt_exp1 = DecisionTreeInterpreter(dt_df, metadata_df, app_id1, model_name1)
dt_exp1.print_tree(as_name=True)
dt_exp2 = DecisionTreeInterpreter(dt_df, metadata_df, app_id2, model_name2)
dt_exp2.print_tree(as_name=True)
dt_exp3 = DecisionTreeInterpreter(dt_df, metadata_df, app_id3, model_name3)
dt_exp3.print_tree(as_name=True)
dt_exp4 = DecisionTreeInterpreter(dt_df, metadata_df, app_id4, model_name4)
dt_exp4.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
lr_exp1 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id1, model_name1)
lr_exp2 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id2, model_name2)
lr_exp3 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id3, model_name3)
lr_exp4 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id4, model_name4)
lr_exp5 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id1, model_name1, variant="sparse")
lr_exp6 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id2, model_name2, variant="sparse")
lr_exp7 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id3, model_name3, variant="sparse")
lr_exp8 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id4, model_name4, variant="sparse")

ai_dataset_loaders = {
    1: ai_data_loader1,
    2: ai_data_loader2,
    3: ai_data_loader3,
    4: ai_data_loader4,
    5: ai_data_loader1,
    6: ai_data_loader2,
    7: ai_data_loader3,
    8: ai_data_loader4
}

lr_exps = {
    1: lr_exp1,     
    2: lr_exp2,
    3: lr_exp3,
    4: lr_exp4,
    5: lr_exp5,     
    6: lr_exp6,
    7: lr_exp7,
    8: lr_exp8
}


Decision Tree (appId=wine_quality, model=mlp)
Fidelity: 0.9550
if Alcohol <= 10.849999904632568:
  if Vinegar Taint <= 0.375:
    if Sulphates <= 0.6049999892711639:
      → Predict 0 (probs: [0.8 0.2])
    else:
      → Predict 1 (probs: [0. 1.])
  else:
    if Sulphates <= 1.64000004529953:
      → Predict 0 (probs: [0.9818 0.0182])
    else:
      → Predict 1 (probs: [0. 1.])
else:
  if Vinegar Taint <= 0.5649999976158142:
    if Sulphates <= 0.5049999952316284:
      → Predict 0 (probs: [0.5 0.5])
    else:
      → Predict 1 (probs: [0. 1.])
  else:
    if Alcohol <= 12.400000095367432:
      → Predict 0 (probs: [0.8889 0.1111])
    else:
      → Predict 1 (probs: [0. 1.])
Decision Tree (appId=mushrooms, model=mlp)
Fidelity: 0.8000
if Height <= 3.4250000715255737:
  if Cap Diameter <= 3.284999966621399:
    if Shape = flat <= 0.5:
      → Predict 0 (probs: [1. 0.])
    else:
      → Predict 0 (probs: [0.6538 0.3462])
  else:
    if Width <= 3.944999933242798:
      → Predict 0 (pro

## Bring in individual strategy RL envs

In [9]:
import importlib
import src.memory as memory
importlib.reload(memory)
import src.dt_memory as dt_memory
importlib.reload(dt_memory)
import src.heuristic_lr_model as heuristic_lr_model
importlib.reload(heuristic_lr_model)
import src.lr_memory as lr_memory
importlib.reload(lr_memory)

from src.memory import DeclarativeMemory, CombinedMemory
from src.dt_memory import (
    add_dt_to_memory, dt_traverse, refresh_dt_path_in_memory
)
from src.heuristic_lr_model import (
    add_lr_heuristic_to_memory, lr_heuristic, refresh_lr_heuristic_in_memory
)
from src.lr_memory import (
    add_lr_calculation_to_memory, lr_calculation, refresh_lr_calculation_in_memory
)

from typing import Optional
import random
from dataclasses import replace

In [10]:
# ---------------- Memory builder ----------------
def _make_memory(retrieval_threshold, latency_factor):
    dm = DeclarativeMemory(
        retrieval_threshold=retrieval_threshold,
        latency_factor=latency_factor,
        latency_exponent=0.5,
        max_assoc_strength=2.0,
        mismatch_penalty=-2.0,
        activation_noise=0.3,
        decay=0.5,
    )
    return CombinedMemory(dm, wm_capacity=7)


def _post_read_refresh(strategy, memory, explainer, *, compute_sf, info_or_aux, instance, with_xai, actual_label, active_indices=None):
    """
    Applies post-read/online refresh depending on strategy.
    - For lr_calc: refresh only when with_xai=True (as before).
    - For lr_heur: refresh on BOTH. If with_xai=False, use model’s own pred as 'actual'.
    - For dt:      after read, refresh the path; retrieve-mode may use refresh_prob_cap internally.
    """
    if strategy == "lr_calc":
        if with_xai:
            refresh_lr_calculation_in_memory(
                memory, explainer,
                intercept_display_sf=int(compute_sf),
                factor_display_sf=int(compute_sf),
                #active_indices=active_indices
            )

    elif strategy == "lr_heur":
        # 'info_or_aux' is the 'info' dict returned by lr_heuristic
        info = info_or_aux
        # actual_label already set by caller to either human response (with XAI) or model pred (w/o XAI)
        refresh_lr_heuristic_in_memory(
            memory, explainer, info, actual=int(actual_label),
            active_indices=active_indices, w_min=1e-4, verbose=False
        )

    elif strategy == "dt":
        if with_xai:
            # refresh the path after reading explanations
            # (thresh_sf ties to compute_sf for consistency)
            refresh_dt_path_in_memory(memory, explainer, instance, thresh_sf=int(compute_sf))

In [11]:
from stable_baselines3 import PPO
import numpy as np

LR_ACTIONS = {
    1: "read",
    2: "retrieve",
}

class HeadlessLRCalcPolicy:
    def __init__(
        self,
        *,
        model_path: str,
        lr_exps,
        memory_factory,
        training_cog_params: dict,
        ddm_a_bins: int = 3,
    ):
        self.name = "lr_calc"
        self.model = PPO.load(model_path)
        self.lr_exps = lr_exps
        self.mem_factory = memory_factory

        self.training_cog_params = dict(training_cog_params)
        self.ddm_a_bins = int(ddm_a_bins)

        # derive ddm_a range from training config (falls back to fixed 1.0)
        ddm_range = self.training_cog_params.get("ddm_a", (1.0, 1.0))
        if isinstance(ddm_range, (list, tuple)) and len(ddm_range) == 2:
            self.ddm_a_min, self.ddm_a_max = float(ddm_range[0]), float(ddm_range[1])
        elif isinstance(ddm_range, (int, float)):
            self.ddm_a_min = self.ddm_a_max = float(ddm_range)
        else:
            self.ddm_a_min = self.ddm_a_max = 1.0

        # remember which cogs must be appended to obs tail (EXACT same rule as training)
        self.obs_tail_cogs = [
            k for k, v in self.training_cog_params.items()
            if (k not in ["chi", "ddm_a"]) and isinstance(v, (list, tuple)) and len(v) == 2
        ]

        # runtime state
        self.memory = None
        self.perm = None
        self.inv_perm = None
        self.with_xai_schedule = None
        self.episode_cogs = {}
        self.episode_len = 0

        self.strategy_counts = np.zeros(2, dtype=np.int32)
        self.strategy_success = np.zeros(2, dtype=np.float32)
        self.contributions = None
        self.step_idx = 0

    def _ddm_a_from_bin(self, b):
        b = int(np.clip(b, 0, self.ddm_a_bins - 1))
        if self.ddm_a_bins == 1 or self.ddm_a_min == self.ddm_a_max:
            return float(self.ddm_a_min)
        frac = (b + 0.5) / self.ddm_a_bins
        return float(self.ddm_a_min + frac * (self.ddm_a_max - self.ddm_a_min))

    def reset(self, *, rng, with_xai_schedule, perm, inv_perm, episode_cogs: dict, dataset_id: int):
        self.step_idx = 0
        self.perm, self.inv_perm = perm, inv_perm
        self.with_xai_schedule = with_xai_schedule
        self.episode_len = int(len(with_xai_schedule))
        self.episode_cogs = dict(episode_cogs)

        self.strategy_counts[:] = 0
        self.strategy_success[:] = 0.0
        self.contributions = {i: [] for i in range(len(self.perm))}

        # memory built from episode cogs (retrieval_threshold / latency_factor etc.)
        rt = float(self.episode_cogs.get("retrieval_threshold", -1.0))
        lf = float(self.episode_cogs.get("latency_factor", 0.0))
        self.memory = self.mem_factory(rt, lf)
        if self.lr_exps.get(dataset_id) is not None:
            add_lr_calculation_to_memory(self.lr_exps[dataset_id], self.memory)
            self.lr_exp = self.lr_exps[dataset_id]
        self.memory.tick(90)

    def _build_obs(self, *, chi_value: float, with_xai_flag: bool):
        # per-feature contribution stats (like LRForward)
        K = len(self.perm)
        stds = np.array([
            (np.std(self.contributions[i]) if len(self.contributions[i]) > 1 else 0.0)
            for i in range(K)
        ], dtype=np.float32)
        stds = (stds / stds.sum()).astype(np.float32) if stds.sum() > 0 else stds

        means = np.array([
            (np.mean(self.contributions[i]) if len(self.contributions[i]) > 0 else 0.0)
            for i in range(K)
        ], dtype=np.float32)
        denom = np.abs(means).sum()
        means = (np.abs(means) / (denom + 1e-9)).astype(np.float32) if denom > 0 else means

        # chi normalization matches your env: chi / chi_high; we need chi_high from training cfg
        chi_high = (
            self.training_cog_params.get("chi", [0.0, 0.03])[1]
            if isinstance(self.training_cog_params.get("chi", None), (list, tuple))
            else max(chi_value, 1e-9)
        )

        base = [
            float(chi_value / max(chi_high, 1e-9)),
            float(self.step_idx / max(self.episode_len, 1)),
            float(with_xai_flag),
            *self.strategy_counts.tolist(),
            *self.strategy_success.tolist(),
            *stds.tolist(),
            *means.tolist(),
        ]

        # Append cogs for keys that had ranges during training (excluding chi, ddm_a)
        for k in self.obs_tail_cogs:
            base.append(float(self.episode_cogs.get(k, 0.0)))

        return np.asarray(base, dtype=np.float32)

    def step(self, *, x_raw, y_true: int, with_xai: bool, chi_value: float, **kwargs):
        # update contributions with current instance and lr_exp coefs (like your env)
        K = len(self.perm)
        for feat in range(K):
            coef = 0.0
            if self.lr_exp is not None and hasattr(self.lr_exp, "coefficients"):
                coef = float(self.lr_exp.coefficients.get(f"a{feat}", 0.0))
            self.contributions[feat].append(float(x_raw[feat]) * coef)

        obs = self._build_obs(chi_value=chi_value, with_xai_flag=with_xai)
        action, _ = self.model.predict(obs, deterministic=True)
        a = np.asarray(action, dtype=np.int64).reshape(-1)

        action_id = int(a[0])              # 0..2; 1="read", 2="retrieve"
        ddm_bin = int(a[1])                # 0..B-1
        mask_bits = a[2:2 + K].tolist()

        chosen_mode = LR_ACTIONS.get(action_id, "retrieve")
        if (not with_xai) and (chosen_mode == "read"):
            # mirror your training legality; simplest is to coerce to retrieve
            chosen_mode = "retrieve"

        active_shuf = [i for i, b in enumerate(mask_bits) if b == 1]
        active_indices = [int(self.perm[i]) for i in active_shuf]
        ddm_a = self._ddm_a_from_bin(ddm_bin)

        # Use episode cogs as defaults for strategy forward
        T_enc = self.episode_cogs.get("T_enc", 2.0)
        T_op  = self.episode_cogs.get("T_op", 0.2)
        ddm_s = self.episode_cogs.get("ddm_s", 1.0)
        compute_sf = int(self.episode_cogs.get("compute_sf", 2))
        lapse = float(self.episode_cogs.get("lapse", 0.0))

        probs, pred_time, _ = lr_calculation(
            x_raw, self.memory, lr_exp=self.lr_exp,
            T_enc=T_enc, T_op=T_op,
            ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf,
            mode=chosen_mode,
            active_indices=active_indices
        )

        if chosen_mode == "read":
            _post_read_refresh(
                "lr_calc", self.memory, self.lr_exp,
                compute_sf=compute_sf,
                info_or_aux=None,
                instance=x_raw,
                with_xai=True,
                actual_label=None,
            )

        # success stats (for obs)
        pr = float(probs[int(y_true)])
        if lapse > 0.0:
            pr = (1.0 - lapse) * pr + 0.5 * lapse

        idx = 0 if (chosen_mode == "read") else 1
        self.strategy_counts[idx] += 1
        n = self.strategy_counts[idx]
        self.strategy_success[idx] = (self.strategy_success[idx] * (n - 1) + (1.0 if pr > 0.5 else 0.0)) / max(n, 1)

        self.step_idx += 1
        return probs, float(pred_time), {
            "mode": chosen_mode,
            "ddm_a": float(ddm_a),
            "ddm_bin": int(ddm_bin),
            "active_indices_orig": active_indices,
            "active_indices_shuf": active_shuf,
            "prob_correct": pr,
        }


In [12]:
from typing import Dict, Any, Optional, Tuple, List
import numpy as np
from stable_baselines3 import PPO

# expects LR_ACTIONS only if you reuse it; here the heuristic policy chooses only [ddm_a_bin, mask_bits]
# from your codebase:
# - _make_memory(rt, lf)
# - add_lr_heuristic_to_memory(lr_exp, memory, initial_var=1.0)
# - lr_heuristic(x_norm, memory, lr_exp, num_samples, K_top, T_enc, ddm_a, ddm_s, ddm_Tnd, ddm_norm, active_indices, verbose)
# - _post_read_refresh(kind, memory, lr_exp, compute_sf, info_or_aux, instance, with_xai, actual_label, active_indices)


class HeadlessLRHeurPolicy:
    """
    Headless wrapper for a trained LR-heuristic agent.

    Action (internal, predicted by the trained PPO):
      [ddm_a_bin, mask_bits[0:F]]

    Observation (recreated to match your LR-heuristic env):
      [chi_norm, trial_norm, with_xai_flag,
       var_norm[0:F], mu_norm[0:F], (ranged cog-params except {chi, ddm_a})]

    Use:
      - Call reset(...) once per episode (outer env controls episode).
      - For each trial, call step(...), passing x_raw, x_norm, y_true, with_xai, chi_value.
    """

    def __init__(
        self,
        *,
        model_path: str,
        lr_exps,
        memory_factory,                      # e.g., _make_memory
        training_cog_params: Dict[str, Any], # the SAME dict used in training
        ddm_a_bins: int = 3,
        heuristic_kwargs: Optional[Dict[str, Any]] = None,  # e.g., {"num_samples":64,"K_top":3}
    ):
        self.name = "lr_heur"
        self.model = PPO.load(model_path)
        self.lr_exps = lr_exps
        self.mem_factory = memory_factory
        self.training_cog_params = dict(training_cog_params)
        self.ddm_a_bins = int(ddm_a_bins)
        self.heuristic_kwargs = dict(heuristic_kwargs or {"num_samples": 64, "K_top": 3})

        # ddm_a range derived from training config
        ddm_spec = self.training_cog_params.get("ddm_a", (1.0, 1.0))
        if isinstance(ddm_spec, (list, tuple)) and len(ddm_spec) == 2:
            self.ddm_a_min, self.ddm_a_max = float(ddm_spec[0]), float(ddm_spec[1])
        elif isinstance(ddm_spec, (int, float)):
            self.ddm_a_min = self.ddm_a_max = float(ddm_spec)
        else:
            self.ddm_a_min = self.ddm_a_max = 1.0

        # which cog-params to append to obs tail (exact same rule as env)
        self.obs_tail_cogs = [
            k for k, v in self.training_cog_params.items()
            if (k not in ["chi", "ddm_a"]) and isinstance(v, (list, tuple)) and len(v) == 2
        ]

        # runtime episode state
        self.memory = None
        self.perm: Optional[np.ndarray] = None
        self.inv_perm: Optional[np.ndarray] = None
        self.with_xai_schedule: Optional[np.ndarray] = None
        self.episode_cogs: Dict[str, Any] = {}
        self.episode_len: int = 0
        self.step_idx: int = 0

    # ---------- internal helpers ----------

    def _ddm_a_from_bin(self, b: int) -> float:
        b = int(np.clip(b, 0, self.ddm_a_bins - 1))
        if self.ddm_a_bins == 1 or self.ddm_a_min == self.ddm_a_max:
            return float(self.ddm_a_min)
        frac = (b + 0.5) / self.ddm_a_bins
        return float(self.ddm_a_min + frac * (self.ddm_a_max - self.ddm_a_min))

    def _read_mu_var_from_memory(self) -> Tuple[np.ndarray, np.ndarray]:
        """Read μ/var per feature from memory chunks and normalize (same as env)."""
        mus, vars_ = [], []
        # assumes lr_exp.coefficients like {"a0": w0, "a1": w1, ...}
        for key in range(self.max_features):
            cname = f"LR_coef_prob_a{key}"
            ch = self.memory.get_chunk(cname)
            mu = ch.slots.get("mu", 0.0) if ch is not None else 0.0
            var = ch.slots.get("var", 0.0) if ch is not None else 0.0
            mus.append(mu)
            vars_.append(var)

        mu_arr = np.array(mus, dtype=np.float32)
        var_arr = np.array(vars_, dtype=np.float32)

        if var_arr.sum() > 0:
            var_norm = (var_arr / var_arr.sum()).astype(np.float32)
        else:
            var_norm = var_arr

        s = np.abs(mu_arr).sum()
        if s > 0:
            mu_norm = (np.abs(mu_arr) / (s + 1e-9)).astype(np.float32)
        else:
            mu_norm = mu_arr

        return mu_norm, var_norm

    def _build_obs(self, *, chi_value: float, with_xai_flag: bool) -> np.ndarray:
        """Exact obs layout used in your LR-heuristic env."""

        mu_norm, var_norm = self._read_mu_var_from_memory()

        # apply permutation used this episode
        if self.perm is not None:
            mu_norm = mu_norm[self.perm]
            var_norm = var_norm[self.perm]

        # chi normalization uses the high from training config
        chi_spec = self.training_cog_params.get("chi", [0.0, 0.03])
        chi_high = float(chi_spec[1]) if isinstance(chi_spec, (list, tuple)) and len(chi_spec) == 2 else max(chi_value, 1e-9)

        base = [
            float(chi_value / max(chi_high, 1e-9)),
            float(self.step_idx / max(self.episode_len, 1)),
            float(with_xai_flag),
            *var_norm.tolist(),
            *mu_norm.tolist(),
        ]

        # append ranged cogs (excluding chi, ddm_a) in the SAME order you trained
        for k in self.obs_tail_cogs:
            base.append(float(self.episode_cogs.get(k, 0.0)))

        return np.asarray(base, dtype=np.float32)

    # ---------- public API ----------

    def reset(
        self,
        *,
        rng,
        with_xai_schedule: np.ndarray,
        perm: np.ndarray,
        inv_perm: np.ndarray,
        episode_cogs: Dict[str, Any],
        dataset_id: int,
    ) -> None:
        """Call once per episode (outer env controls sampling & schedules)."""
        self.step_idx = 0
        self.with_xai_schedule = with_xai_schedule
        self.episode_len = int(len(with_xai_schedule))
        self.perm, self.inv_perm = perm, inv_perm
        self.episode_cogs = dict(episode_cogs)

        # build memory from episode-varying cogs
        rt = float(self.episode_cogs.get("retrieval_threshold", -1.0))
        lf = float(self.episode_cogs.get("latency_factor", 0.0))
        self.memory = self.mem_factory(rt, lf)

        # seed heuristic “knowledge”
        if self.lr_exps.get(dataset_id) is not None:
            self.lr_exp = self.lr_exps[dataset_id]
            add_lr_heuristic_to_memory(self.lr_exp, self.memory, initial_var=0.1)

        self.memory.tick(90)

    def step(
        self,
        *,
        x_raw: np.ndarray,
        x_norm: np.ndarray,
        y_true: int,
        with_xai: bool,
        chi_value: float,
    ) -> Tuple[np.ndarray, float, Dict[str, Any]]:
        """
        Drive ONE trial through the trained heuristic policy.
        Returns: (probs, pred_time, info)
        """
        self.max_features = x_raw.shape[-1]

        # 1) Build the observation expected by the trained policy
        obs = self._build_obs(chi_value=chi_value, with_xai_flag=with_xai)

        # 2) Policy picks [ddm_a_bin, mask_bits...]
        action, _ = self.model.predict(obs, deterministic=True)
        a = np.asarray(action, dtype=np.int64).reshape(-1)
        ddm_a_bin = int(a[0])
        K = len(self.perm)
        mask_bits = a[1:1 + K].tolist()

        active_shuf = [i for i, b in enumerate(mask_bits) if b == 1]
        active_indices = [int(self.perm[i]) for i in active_shuf]
        ddm_a = self._ddm_a_from_bin(ddm_a_bin)

        # 3) Pull defaults from episode cogs (same names as your env)
        T_enc     = float(self.episode_cogs.get("T_enc", 2.0))
        ddm_s     = float(self.episode_cogs.get("ddm_s", 1.0))
        ddm_Tnd   = float(self.episode_cogs.get("ddm_Tnd", 0.30))
        ddm_norm  = self.episode_cogs.get("ddm_norm", "l2")
        compute_sf = int(self.episode_cogs.get("compute_sf", 2))
        lapse     = float(self.episode_cogs.get("lapse", 0.0))

        # 4) Run heuristic LR
        probs, pred_time, aux = lr_heuristic(
            x_norm, self.memory, self.lr_exp,
            num_samples=int(self.heuristic_kwargs.get("num_samples", 64)),
            K_top=int(self.heuristic_kwargs.get("K_top", 3)),
            T_enc=T_enc,
            ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=ddm_Tnd, ddm_norm=ddm_norm,
            active_indices=active_indices,
            verbose=False
        )

        # 5) Post-read refresh/learning (uses with_xai to modulate learning)
        _post_read_refresh(
            "lr_heur", self.memory, self.lr_exp,
            compute_sf=compute_sf,
            info_or_aux=aux,
            instance=x_raw,
            with_xai=with_xai,
            actual_label=int(y_true),
            active_indices=active_indices
        )

        # (Optional) apply lapse here if you want prob_correct for logging
        prob_correct = float(probs[int(y_true)])
        if lapse > 0.0:
            prob_correct = (1.0 - lapse) * prob_correct + 0.5 * lapse

        self.step_idx += 1
        return probs, float(pred_time), {
            "ddm_a": float(ddm_a),
            "ddm_a_bin": int(ddm_a_bin),
            "active_indices_orig": active_indices,
            "active_indices_shuf": active_shuf,
            "prob_correct": prob_correct,
        }


In [13]:
from typing import Any, Dict, Optional, Tuple
import numpy as np
from stable_baselines3 import PPO

# expected user-provided funcs/objects:
# - _make_memory(retrieval_threshold, latency_factor)
# - add_dt_to_memory(memory, dt_explainer)   # optional helper; skip if not needed
# - dt_traverse(instance, memory, explainer, ...)
# - _post_read_refresh("dt", memory, explainer, ...)
# - your dt explainers dict: dt_exps

class HeadlessDTPolicy:
    """
    Headless wrapper for a trained DT-forward agent (DTForward).

    Trained policy action: [strategy_id, ddm_a_bin]
      strategy_id in {0,1,2}  -> 0 invalid, 1="read", 2="retrieve"
      ddm_a_bin in {0..B-1}

    Observation (recreated to match DTForward env):
      [chi_norm, trial_norm, with_xai,
       count_read, count_retrieve,
       succ_read, succ_retrieve]

    Usage:
      - reset(...) once per episode (outer driver sets with_xai_schedule, episode cogs, dataset id)
      - step(...) per trial with (x_raw, y_true, with_xai, chi_value)
    """

    def __init__(
        self,
        *,
        model_path: str,
        dt_exps: Dict[int, Any],
        memory_factory,                         # e.g., _make_memory
        training_cog_params: Dict[str, Any],    # SAME dict used during training
        ddm_a_bins: int = 3,
        forbid_read_without_xai: bool = True,   # align with env illegality
        dt_kwargs: Optional[Dict[str, Any]] = None,  # e.g., {"n_mc": 64, "topk_k": 3}
    ):
        self.name = "dt"
        self.model = PPO.load(model_path)
        self.dt_exps = dt_exps
        self.mem_factory = memory_factory
        self.training_cog_params = dict(training_cog_params)
        self.ddm_a_bins = int(ddm_a_bins)
        self.forbid_read_without_xai = bool(forbid_read_without_xai)
        self.dt_kwargs = dict(dt_kwargs or {"n_mc": 64, "topk_k": 3, "refresh_prob_cap": 1.0})

        # ddm_a range (from training schema)
        ddm_spec = self.training_cog_params.get("ddm_a", (1.0, 1.0))
        if isinstance(ddm_spec, (list, tuple)) and len(ddm_spec) == 2:
            self.ddm_a_min, self.ddm_a_max = float(ddm_spec[0]), float(ddm_spec[1])
        elif isinstance(ddm_spec, (int, float)):
            self.ddm_a_min = self.ddm_a_max = float(ddm_spec)
        else:
            self.ddm_a_min = self.ddm_a_max = 1.0

        # chi normalization reference
        chi_spec = self.training_cog_params.get("chi", [0.0, 0.03])
        self.chi_high = float(chi_spec[1]) if isinstance(chi_spec, (list, tuple)) and len(chi_spec) == 2 else 0.03

        # which ranged cogs exist is irrelevant for obs (DT obs is compact); we still store episode cogs
        self.episode_cogs: Dict[str, Any] = {}

        # episode runtime state
        self.memory = None
        self.with_xai_schedule: Optional[np.ndarray] = None
        self.episode_len: int = 0
        self.step_idx: int = 0
        self.strategy_counts = np.zeros(2, dtype=np.int32)   # [read, retrieve]
        self.strategy_success = np.zeros(2, dtype=np.float32)
        self.dt_exp = None

    # ---------- helpers ----------

    def _ddm_a_from_bin(self, b: int) -> float:
        b = int(np.clip(b, 0, self.ddm_a_bins - 1))
        if self.ddm_a_bins == 1 or self.ddm_a_min == self.ddm_a_max:
            return float(self.ddm_a_min)
        frac = (b + 0.5) / self.ddm_a_bins
        return float(self.ddm_a_min + frac * (self.ddm_a_max - self.ddm_a_min))

    def _build_obs(self, *, chi_value: float, with_xai: bool) -> np.ndarray:
        # Observation layout identical to DTForward
        obs = np.array([
            float(chi_value / max(self.chi_high, 1e-9)),
            float(self.step_idx / max(self.episode_len, 1)),
            float(with_xai),
            float(self.strategy_counts[0]),   # read count
            float(self.strategy_counts[1]),   # retrieve count
            float(self.strategy_success[0]),  # read success rate
            float(self.strategy_success[1]),  # retrieve success rate
        ], dtype=np.float32)
        return obs

    def _decode_action(self, action) -> Tuple[int, int]:
        a = np.asarray(action)
        if a.ndim == 0:
            a = np.array([int(a)], dtype=int)
        if a.ndim > 1:
            a = a[0]
        a = a.astype(int).ravel()
        if a.size < 2:
            out = np.zeros(2, dtype=int)
            out[:a.size] = a
            a = out
        return int(a[0]), int(a[1])

    # ---------- public API ----------

    def reset(
        self,
        *,
        rng,
        with_xai_schedule: np.ndarray,
        episode_cogs: Dict[str, Any],
        dataset_id: int,
        **kwargs
    ) -> None:
        """
        Call once per episode. Outer driver supplies the schedule and episode cogs.
        """
        self.step_idx = 0
        self.with_xai_schedule = np.asarray(with_xai_schedule, dtype=bool)
        self.episode_len = int(len(self.with_xai_schedule))
        self.episode_cogs = dict(episode_cogs)
        self.strategy_counts[:] = 0
        self.strategy_success[:] = 0.0

        # Build memory from cogs
        rt = float(self.episode_cogs.get("retrieval_threshold", 0.5))
        lf = float(self.episode_cogs.get("latency_factor", 3.0))
        self.memory = self.mem_factory(rt, lf)

        # Attach DT explainer (per dataset)
        self.dt_exp = self.dt_exps.get(dataset_id, None)
        if self.dt_exp is not None:
            try:
                add_dt_to_memory(self.memory, self.dt_exp)  # optional helper
            except NameError:
                pass
        self.memory.tick(90)

    def step(
        self,
        *,
        x_raw: np.ndarray,
        y_true: int,
        with_xai: bool,
        chi_value: float,
        **kwargs
    ) -> Tuple[np.ndarray, float, Dict[str, Any]]:
        """
        Drive ONE trial with the trained DT policy.
        Returns: (probs, pred_time, info)
        """
        # 1) Build observation for the trained policy
        obs = self._build_obs(chi_value=chi_value, with_xai=with_xai)

        # 2) Policy selects [strategy_id, ddm_a_bin]
        action, _ = self.model.predict(obs, deterministic=True)
        strategy_id, ddm_a_bin = self._decode_action(action)

        # 3) Validate/adjust chosen mode (read not allowed without XAI if forbid flag)
        chosen_mode = {1: "read", 2: "retrieve"}.get(strategy_id, "invalid")
        illegal = False
        if chosen_mode == "invalid":
            illegal = True
            chosen_mode = "retrieve"  # safe fallback
        if self.forbid_read_without_xai and (not with_xai) and (chosen_mode == "read"):
            illegal = True
            chosen_mode = "retrieve"  # force legal action

        # 4) Params (ddm_a from action; others from episode cogs)
        ddm_a   = self._ddm_a_from_bin(ddm_a_bin)
        T_enc   = float(self.episode_cogs.get("T_enc", 2.0))
        ddm_s   = float(self.episode_cogs.get("ddm_s", 1.0))
        ddm_Tnd = float(self.episode_cogs.get("ddm_Tnd", 0.30))
        ddm_norm = self.episode_cogs.get("ddm_norm", "l2")
        compute_sf = int(self.episode_cogs.get("compute_sf", 2))
        lapse   = float(self.episode_cogs.get("lapse", 0.05))

        # 5) DT forward
        probs, pred_time, aux = dt_traverse(
            x_raw, self.memory, self.dt_exp,
            mode=chosen_mode,
            compute_sf=compute_sf,
            T_enc=T_enc, ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=ddm_Tnd, ddm_norm=ddm_norm,
            n_mc=int(self.dt_kwargs.get("n_mc", 64)),
            topk_k=int(self.dt_kwargs.get("topk_k", 3)),
            refresh_prob_cap=float(self.dt_kwargs.get("refresh_prob_cap", 1.0)),
            verbose=False
        )

        # 6) Reward ingredients (same as env)
        prob_correct = float(probs[int(y_true)])
        if lapse > 0.0:
            prob_correct = (1.0 - lapse) * prob_correct + 0.5 * lapse

        # You can compute reward here if needed by the caller:
        # reward = prob_correct - chi_value * float(pred_time)

        # 7) Update success stats by chosen mode
        idx = 0 if (chosen_mode == "read") else 1
        self.strategy_counts[idx] += 1
        n = int(self.strategy_counts[idx])
        old = float(self.strategy_success[idx])
        self.strategy_success[idx] = (old * (n - 1) + (1.0 if prob_correct > 0.5 else 0.0)) / max(n, 1)

        # 8) Post-read refresh (XAI availability controls path-refresh policy)
        _post_read_refresh(
            "dt", self.memory, self.dt_exp,
            compute_sf=compute_sf,
            info_or_aux=aux,
            instance=x_raw,
            with_xai=with_xai,       # availability, not chosen mode
            actual_label=int(y_true)
        )

        self.step_idx += 1
        return probs, float(pred_time), {
            "illegal_action": illegal,
            "chosen_mode": chosen_mode,
            "ddm_a": float(ddm_a),
            "ddm_a_bin": int(ddm_a_bin),
            "prob_correct": prob_correct,
            "counts": self.strategy_counts.copy(),
            "success": self.strategy_success.copy(),
        }


## Make the overall strategy environment

In [142]:
from typing import Dict, Any, Optional, Sequence, Tuple
import numpy as np
from stable_baselines3 import PPO

# --- canonical names & helpers (align with your env) ---
STRAT_DT = "dt"
STRAT_LR_CALC = "lr_calc"
STRAT_LR_HEUR = "lr_heur"
LR_FAMILY = {STRAT_LR_CALC, STRAT_LR_HEUR}

COND_DT, COND_LR, COND_DTLR = "DT", "LR", "DT+LR"
TYPE_DT, TYPE_LR = "DT", "LR"

def _build_with_xai_schedule(N: int, ratio: float, rng: np.random.Generator) -> np.ndarray:
    k = int(round(N * ratio))
    flags = np.array([1] * k + [0] * (N - k), dtype=np.int32)
    rng.shuffle(flags)
    return flags.astype(bool)

def _build_trial_type_schedule(N: int, condition: str, rng: np.random.Generator) -> np.ndarray:
    if condition == COND_DT:
        return np.array([TYPE_DT] * N, dtype=object)
    if condition == COND_LR:
        return np.array([TYPE_LR] * N, dtype=object)
    # DT+LR: half/half shuffled (rounded down/up)
    m = N // 2
    arr = np.array([TYPE_DT] * m + [TYPE_LR] * (N - m), dtype=object)
    rng.shuffle(arr)
    return arr

def _onehot_condition(condition: str) -> np.ndarray:
    return np.array([
        1.0 if condition == COND_DT else 0.0,
        1.0 if condition == COND_LR else 0.0,
        1.0 if condition == COND_DTLR else 0.0,
    ], dtype=np.float32)

def _onehot_trial_type(tt: str) -> np.ndarray:
    return np.array([
        1.0 if tt == TYPE_DT else 0.0,
        1.0 if tt == TYPE_LR else 0.0,
    ], dtype=np.float32)

def _strategy_allowed_under_condition(condition: str, strat_name: str) -> bool:
    if condition == COND_DT:
        return strat_name == STRAT_DT
    if condition == COND_LR:
        return strat_name in LR_FAMILY
    return True  # DT+LR
def run_meta_on_batch(
    *,
    meta_model: PPO,
    strategies: Dict[str, Any],                       # {"dt": ..., "lr_calc": ..., "lr_heur": ...}
    strategy_order: Optional[Sequence[str]] = None,

    # --- Episode data you provide ---
    X_raw: np.ndarray,                                # (N, F_raw)
    y_raw: np.ndarray,                                # (N,)
    X_norm: Optional[np.ndarray] = None,              # (N, F_norm)
    with_xai_schedule: Optional[np.ndarray] = None,   # length N (bool); else derived from ratio
    with_xai_ratio: Optional[float] = None,           # only if schedule is None
    trial_type_schedule: Optional[np.ndarray] = None, # length N of {"DT","LR"}; else derived from condition
    condition: str = COND_DTLR,                       # "DT", "LR", or "DT+LR"
    perm: Optional[np.ndarray] = None,
    dataset_id: Optional[int] = None,

    # --- Cognition controls you provide ---
    episode_cogs: Dict[str, Any],
    training_cog_params: Dict[str, Any],
    chi_value: float,

    # --- Misc ---
    deterministic: bool = False,
    rng: Optional[np.random.Generator] = None,
    invalid_action_penalty: float = -1.0,
) -> Dict[str, Any]:
    """
    Runs ONE episode (length N = len(X_raw)) with episode-level 'condition' and per-trial 'trial_type'.

    Observation to meta matches the training env:
      [chi_norm, trial_norm, with_xai, cond_onehot(3), trial_type_onehot(2),
       per-strategy stats (4*S): for each strategy in strategy_order →
         [count_with/N, mean_with, count_without/N, mean_without]]
    """
    assert X_raw.ndim == 2, "X_raw must be (N, F)"
    N = X_raw.shape[0]
    if X_norm is None:
        X_norm = X_raw
    assert X_norm.shape[0] == N
    assert y_raw.shape[0] == N
    assert condition in {COND_DT, COND_LR, COND_DTLR}, f"Unknown condition: {condition}"

    if rng is None:
        rng = np.random.default_rng()

    # --- strategy ordering / mapping
    if strategy_order is None:
        strategy_order = list(strategies.keys())
    S = len(strategy_order)
    name_from_idx = {i: strategy_order[i] for i in range(S)}

    # --- with-XAI schedule
    if with_xai_schedule is None:
        ratio = float(with_xai_ratio if with_xai_ratio is not None else 0.5)
        with_xai_schedule = _build_with_xai_schedule(N, ratio, rng)
    else:
        with_xai_schedule = np.asarray(with_xai_schedule, dtype=bool)
        assert len(with_xai_schedule) == N

    # --- trial type schedule (DT/LR)
    if trial_type_schedule is None:
        trial_type_schedule = _build_trial_type_schedule(N, condition, rng)
    else:
        trial_type_schedule = np.asarray(trial_type_schedule, dtype=object)
        assert len(trial_type_schedule) == N
        assert set(trial_type_schedule.tolist()) <= {TYPE_DT, TYPE_LR}, "trial_type_schedule must contain only 'DT'/'LR'"

    # --- feature permutation
    F = X_raw.shape[1]
    if perm is None:
        perm = np.arange(F, dtype=np.int64)
    perm = np.asarray(perm, dtype=np.int64)
    assert perm.shape[0] == F
    inv_perm = np.empty_like(perm); inv_perm[perm] = np.arange(F, dtype=np.int64)

    # --- chi normalization ref (match training)
    chi_spec = training_cog_params.get("chi", [0.0, 0.03])
    if isinstance(chi_spec, (list, tuple)) and len(chi_spec) == 2:
        chi_high = float(chi_spec[1])
    elif isinstance(chi_spec, (int, float)):
        chi_high = float(chi_spec)
    else:
        chi_high = 0.03
    chi_high = max(chi_high, 1e-9)
    chi_norm = float(chi_value / chi_high)

    # --- reset strategies with shared context
    shared_reset = dict(
        rng=rng,
        with_xai_schedule=with_xai_schedule,
        perm=perm,
        inv_perm=inv_perm,
        episode_cogs=dict(episode_cogs),
        dataset_id=(dataset_id if dataset_id is not None else 1),
    )
    for sname in strategy_order:
        strategies[sname].reset(**shared_reset)

    # --- per-episode stats (to mirror env)
    stats = {
        name: {
            "with": {"count": 0, "sum_pr": 0.0},
            "without": {"count": 0, "sum_pr": 0.0},
        }
        for name in strategy_order
    }
    denom_N = float(max(1, N))

    def _stats_vector() -> np.ndarray:
        out = []
        for name in strategy_order:
            w = stats[name]["with"]
            wo = stats[name]["without"]
            count_w = float(w["count"]); count_wo = float(wo["count"])
            mean_w = (w["sum_pr"] / count_w) if count_w > 0 else 0.0
            mean_wo = (wo["sum_pr"] / count_wo) if count_wo > 0 else 0.0
            out.extend([
                count_w / denom_N,
                float(mean_w),
                count_wo / denom_N,
                float(mean_wo),
            ])
        return np.asarray(out, dtype=np.float32)

    # --- run
    total_reward = 0.0
    logs = {
        "strategy_name": [], "action_idx": [],
        "with_xai_requested": [], "with_xai_used": [],
        "trial_type": [], "condition": [], "mismatch_applied": [],
        "invalid_under_condition": [],
        "prob_correct": [], "probs": [], "pred_time": [], "reward": [], "info": [],
    }

    cond_oh = _onehot_condition(condition)

    for t in range(N):
        with_xai_req = bool(with_xai_schedule[t])
        trial_type = str(trial_type_schedule[t])  # "DT" or "LR"

        # === Build observation exactly like the env ===
        obs = np.concatenate([
            np.array([chi_norm, float(t / N), float(with_xai_req)], dtype=np.float32),
            cond_oh,
            _onehot_trial_type(trial_type),
            _stats_vector(),   # <-- NEW: append per-strategy stats
        ]).astype(np.float32)

        action, _ = meta_model.predict(obs, deterministic=False)
        a = int(action) if (0 <= int(action) < S) else 0
        sname = name_from_idx[a]

        random_num = rng.uniform(0.0, 1.0)
        if random_num < 0.9:
            sname = 'lr_calc' if 'lr' in sname else sname  # normalize 'lr' to 'lr_calc'

        # condition gating: penalize disallowed strategies
        if not _strategy_allowed_under_condition(condition, sname):
            reward = float(invalid_action_penalty)
            total_reward += reward
            logs["strategy_name"].append(sname)
            logs["action_idx"].append(a)
            logs["with_xai_requested"].append(with_xai_req)
            logs["with_xai_used"].append(False)
            logs["trial_type"].append(trial_type)
            logs["condition"].append(condition)
            logs["mismatch_applied"].append(False)
            logs["invalid_under_condition"].append(True)
            logs["prob_correct"].append(0.0)
            logs["pred_time"].append(0.0)
            logs["probs"].append([0.0, 0.0])
            logs["reward"].append(reward)
            logs["info"].append({"invalid_under_condition": True})
            continue



        strat = strategies[sname]

        # mismatch logic: WITH-XAI but wrong family -> run WITHOUT-XAI
        with_xai_used = with_xai_req
        mismatch = False
        if with_xai_req:
            if trial_type == TYPE_DT and sname in LR_FAMILY:
                with_xai_used = False; mismatch = True
            elif trial_type == TYPE_LR and sname == STRAT_DT:
                with_xai_used = False; mismatch = True

        probs, pred_time, info = strat.step(
            x_raw=X_raw[t],
            x_norm=X_norm[t],
            y_true=int(y_raw[t]),
            with_xai=with_xai_used,
            chi_value=float(chi_value),
        )

        pr = float(probs[int(y_raw[t])])
        reward = pr - float(chi_value) * float(pred_time)
        total_reward += reward

        # === Update per-episode stats (must mirror env) ===
        mode_key = "with" if with_xai_used else "without"
        entry = stats[sname][mode_key]
        entry["count"] += 1
        entry["sum_pr"] += pr

        # logs
        logs["strategy_name"].append(sname)
        logs["action_idx"].append(a)
        logs["with_xai_requested"].append(with_xai_req)
        logs["with_xai_used"].append(with_xai_used)
        logs["trial_type"].append(trial_type)
        logs["condition"].append(condition)
        logs["mismatch_applied"].append(mismatch)
        logs["invalid_under_condition"].append(False)
        logs["prob_correct"].append(pr)
        logs["pred_time"].append(float(pred_time))
        logs["probs"].append(probs.tolist())
        logs["reward"].append(float(reward))
        logs["info"].append(info or {})

    return {
        "total_reward": float(total_reward),
        "mean_reward": float(np.mean(logs["reward"])) if N > 0 else 0.0,
        "logs": logs,
        "meta": {
            "N": N,
            "chi_value": float(chi_value),
            "chi_high": float(chi_high),
            "strategy_order": list(strategy_order),
            "dataset_id": dataset_id,
            "episode_cogs": dict(episode_cogs),
            "condition": condition,
        },
    }



In [106]:
training_cog_params = {
    # Memory (used by _make_memory)
    "retrieval_threshold": [-2.0, 0.5],
    "latency_factor": [0.0, 0.5], #[1.0, 5.0],

    # lr_calculation parameters
    "T_enc": [0.5, 3.0], #[0.5, 3.0], #[1.0, 3.0],
    "T_op": [1.0, 3.0], #[0.05, 0.5],
    "ddm_a": [0.6, 1.7], #[0.8, 1.4],
    "ddm_s": [0.7, 1.1], #[0.8, 1.2], #[0.5, 1.2],
    "ddm_Tnd": 0.30,      # fixed is fine
    "ddm_norm": "l2",     # fixed is fine
    "compute_sf": 2.0, #[1.0, 3.0],
    "lapse": 0.05, #[0.01, 0.2],

    "chi": [0.0, 0.02], #[0.0, 0.03],
}

## Fit to participant data

In [107]:
import math
from dataclasses import dataclass
from typing import Dict, Any, Optional, Sequence, Tuple
import numpy as np

# --- safe prob selection for "probability of the RESPONSE" (not correctness) ---
def _select_prob_of_response(probs, y_resp, eps=1e-4) -> float:
    if hasattr(probs, "__len__") and len(probs) == 2:
        return float(probs[1]) if float(y_resp) > 0 else float(probs[0])
    raise ValueError(f"Unexpected probs format: {probs}")

def _bernoulli_nll_from_response_probs(p_list):
    return float(np.sum([-math.log(p) for p in p_list]))

def _mae_time_qtrim(rt_true, rt_pred, q=0.95) -> float:
    if len(rt_true) == 0:
        return 0.0
    t = np.asarray(rt_true, float); p = np.asarray(rt_pred, float)
    thresh = np.quantile(t, q)
    mask = t <= thresh
    if not np.any(mask):
        return 0.0
    return float(np.mean(np.abs(t[mask] - p[mask])))

def _apply_lapse(p, lapse: float) -> float:
    # Mixes with uniform over 2 labels (binary); adjust if multi-class K!=2
    if lapse <= 0.0: 
        return p
    return (1.0 - lapse) * p + 0.5 * lapse  # for binary responses


In [143]:
def meta_episode_likelihood(
    *,
    meta_model,
    strategies: Dict[str, Any],
    strategy_order: Optional[Sequence[str]],
    X_raw: np.ndarray,           # (N, F)
    y_raw: np.ndarray,          # (N,) participant RESPONSES (0/1 for binary)
    X_norm: Optional[np.ndarray],
    response: np.ndarray,        # Participant responses
    with_xai_schedule: Optional[np.ndarray],
    with_xai_ratio: Optional[float],
    trial_type_schedule: Optional[np.ndarray],
    condition: str,
    perm: Optional[np.ndarray],
    dataset_id: int,
    # cognition
    episode_cogs: Dict[str, Any],
    training_cog_params: Dict[str, Any],
    chi_value: float,
    # options
    deterministic: bool = True,
    rng: Optional[np.random.Generator] = None,
    # timing loss config
    time_q: float = 0.90,
    time_weight: float = 0.0,        # set >0 to include timing
    lapse_key: str = "lapse",        # looks inside episode_cogs for lapse
    **kwargs
) -> Dict[str, Any]:
    """
    Runs ONE meta-episode and returns likelihood components **w.r.t. participant responses**.
    - Does NOT use ground-truth labels anywhere for the NLL.
    - Uses lapse from episode_cogs if present (binary uniform mixing).
    - Optionally adds outlier-trimmed MAE on times.
    """
    # We can pass any dummy vector for y_true into run_meta_on_batch, because
    # we only care about the returned probs per trial. To be safest, pass zeros.
    out = run_meta_on_batch(
        meta_model=meta_model,
        strategies=strategies,
        strategy_order=strategy_order,
        X_raw=X_raw,
        y_raw=y_raw,                  # <-- not used for our likelihood
        X_norm=X_norm,
        with_xai_schedule=with_xai_schedule,
        with_xai_ratio=with_xai_ratio,
        trial_type_schedule=trial_type_schedule,
        condition=condition,
        perm=perm,
        dataset_id=dataset_id,
        episode_cogs=episode_cogs,
        training_cog_params=training_cog_params,
        chi_value=chi_value,
        deterministic=deterministic,
        rng=rng,
    )

    logs = out["logs"]
    # Each entry is probs for all classes; we pick participant's chosen label
    probs_list = logs["probs"]             # list of length N, each is list[2] for binary
    pred_time  = np.asarray(logs["pred_time"], float)
    # If you logged actual participant RTs elsewhere, pass them in instead.
    # Here we assume you have them parallel to X_raw:
    #   -> supply externally or attach them to run_meta_on_batch if needed.
    # For now, we default to zeros to keep function standalone.
    rt_true = np.zeros_like(pred_time)

    lapse = float(episode_cogs.get(lapse_key, 0.0))
    p_resp = []
    for t in range(len(probs_list)):
        p = _select_prob_of_response(probs_list[t], response[t])
        p = _apply_lapse(p, lapse)  # optional lapse mixing (binary)
        p_resp.append(p)

    nll = _bernoulli_nll_from_response_probs(p_resp)
    mae_time = _mae_time_qtrim(rt_true, pred_time, q=time_q) if time_weight > 0 else 0.0
    total = nll + time_weight * mae_time

    return {
        "total": total,
        "respNLL": nll,
        "timeMAE": mae_time,
        "p_resp": np.array(p_resp, float),
        "pred_time": pred_time,
        "meta_raw": out,  # keep original logs if you want
        "probs": probs_list,
        "strategies": logs["strategy_name"],

        "condition": condition,
        "trial_type": logs["trial_type"],
        "with_xai_used": logs["with_xai_used"],
        "with_xai_requested": logs["with_xai_requested"],
        "mismatch_applied": logs["mismatch_applied"],
        "invalid_under_condition": logs["invalid_under_condition"],
    }
from skopt import gp_minimize
from skopt.space import Real

# Which cog params should be log-scaled
_LOG_KEYS = {"T_enc", "T_op", "latency_factor", "lapse"}  # adapt to your cogs

@dataclass
class MetaFitConfig:
    # weights
    w_time: float = 0.0          # multiply onto MAE (NLL has implicit weight 1)
    time_q: float = 0.90
    # bounds
    bounds: Dict[str, Tuple[float, float]] = None
    # initial guess
    init_vals: Dict[str, float] = None
    # BO knobs
    n_calls: int = 40
    n_initial_points: int = 8
    random_state: int = 0
    deterministic: bool = True

def _to_opt_coords(name: str, v: float) -> float:
    if name in _LOG_KEYS:
        if v <= 0:
            raise ValueError(f"log transform not defined for {name}={v} (must be >0).")
        return math.log(v)
    return float(v)


def _from_opt_coords(name: str, v: float) -> float:
    return float(np.exp(v)) if name in _LOG_KEYS else float(v)


def fit_meta_params_gp_bo(
    *,
    # meta run stuff
    meta_model,
    strategies: Dict[str, Any],
    strategy_order: Optional[Sequence[str]],
    X_raw: np.ndarray,
    y_raw: np.ndarray,
    X_norm: Optional[np.ndarray],
    response: np.ndarray,
    with_xai_schedule: Optional[np.ndarray],
    with_xai_ratio: Optional[float],
    trial_type_schedule: Optional[np.ndarray],
    condition: str,
    perm: Optional[np.ndarray],
    dataset_id: int,
    training_cog_params: Dict[str, Any],

    # what to tune
    tune_keys: Sequence[str],                     # e.g. ["retrieval_threshold","latency_factor","ddm_a","ddm_s","lapse","chi_value"]
    freeze: Optional[Dict[str, float]] = None,    # keys -> fixed value
    base_episode_cogs: Optional[Dict[str, Any]] = None,

    # config
    cfg: MetaFitConfig = None,

    invalid_action_penalty: float = -1.0,   # passed to meta run
):
    freeze = dict(freeze or {})
    base_episode_cogs = dict(base_episode_cogs or {})
    cfg = cfg or MetaFitConfig()

    # default bounds/initials if not provided
    default_bounds = {
        "T_enc": (0.05, 10.0),
        "T_op": (0.05, 5.0),
        "retrieval_threshold": (-2.0, 1.5),
        "latency_factor": (0.001, 3.0),
        "ddm_a": (0.5, 3.0),
        "ddm_s": (0.5, 2.0),
        "lapse": (0.0, 0.3),
        "chi_value": (0.0, 0.05),
    }
    bounds = dict(default_bounds)
    if cfg.bounds:
        bounds.update(cfg.bounds)

    init_vals = {
        "T_enc": 1.5,
        "T_op": 0.3,
        "retrieval_threshold": -0.5,
        "latency_factor": 0.3,
        "ddm_a": 1.0,
        "ddm_s": 1.0,
        "lapse": 0.05,
        "chi_value": 0.01,
    }
    if cfg.init_vals:
        init_vals.update(cfg.init_vals)

    # Build skopt space (skip frozen)
    space, names = [], []
    for k in tune_keys:
        if k in freeze:
            continue
        lo, hi = bounds[k]
        lo_opt, hi_opt = _to_opt_coords(k, lo), _to_opt_coords(k, hi)
        space.append(Real(lo_opt, hi_opt, name=k))
        names.append(k)

    def _expand_params(x_vec_opt):
        # Start with frozen/base
        p = dict(base_episode_cogs)
        chi = freeze.get("chi_value", init_vals["chi_value"])
        # Insert tuned values
        for k, v_opt in zip(names, x_vec_opt):
            v = _from_opt_coords(k, v_opt)
            if k == "chi_value":
                chi = v
            else:
                p[k] = v
        # Fill in non-tuned defaults if missing
        for k in tune_keys:
            if k == "chi_value":
                continue
            if k not in p:
                p[k] = freeze.get(k, init_vals[k])
        return p, chi

    # Initial point
    x0 = []
    for k in names:
        base = freeze.get(k, init_vals[k])
        x0.append(_to_opt_coords(k, base))

    eval_log = []

    def f_obj(x_vec_opt):
        episode_cogs, chi_value = _expand_params(x_vec_opt)
        res = meta_episode_likelihood(
            meta_model=meta_model,
            strategies=strategies,
            strategy_order=strategy_order,
            X_raw=X_raw,
            y_raw=y_raw,
            X_norm=X_norm,
            response=response,
            with_xai_schedule=with_xai_schedule,
            with_xai_ratio=with_xai_ratio,
            trial_type_schedule=trial_type_schedule,
            condition=condition,
            perm=perm,
            dataset_id=dataset_id,
            episode_cogs=episode_cogs,
            training_cog_params=training_cog_params,
            chi_value=chi_value,
            deterministic=cfg.deterministic,
            invalid_action_penalty=invalid_action_penalty,
            time_q=cfg.time_q,
            time_weight=cfg.w_time,
        )
        eval_log.append({"y": res["total"], **episode_cogs, "chi_value": chi_value, **res})
        print(f"Eval {len(eval_log)}: total={res['total']:.4f}, respNLL={res['respNLL']:.4f}, timeMAE={res['timeMAE']:.4f}, cogs={episode_cogs}, chi_value={chi_value:.4f}")

        return res["total"]

    # If nothing is tunable, just evaluate once
    if not space:
        episode_cogs, chi_value = _expand_params([])
        res = meta_episode_likelihood(
            meta_model=meta_model, strategies=strategies, strategy_order=strategy_order,
            X_raw=X_raw, y_raw=y_raw, X_norm=X_norm, response=response,
            with_xai_schedule=with_xai_schedule, with_xai_ratio=with_xai_ratio,
            trial_type_schedule=trial_type_schedule, condition=condition,
            perm=perm, dataset_id=dataset_id,
            episode_cogs=episode_cogs, training_cog_params=training_cog_params,
            chi_value=chi_value, deterministic=cfg.deterministic, invalid_action_penalty=invalid_action_penalty,
            time_q=cfg.time_q, time_weight=cfg.w_time
        )
        return {
            "best_params": {"episode_cogs": episode_cogs, "chi_value": chi_value},
            "objective": res["total"],
            "respNLL": res["respNLL"],
            "timeMAE": res["timeMAE"],
            "optimizer": "GP(skopt) (all frozen)",
            "n_evals": 1,
            "history": eval_log,
        }

    # Seed with x0
    y0 = f_obj(x0)

    res = gp_minimize(
        f_obj, space,
        x0=[x0], y0=[y0],
        n_calls=cfg.n_calls,
        n_initial_points=cfg.n_initial_points,
        acq_func="EI",
        random_state=cfg.random_state,
        noise=1e-6,
    )


    # === Pick the best ACTUAL evaluation from eval_log (no re-run) ===
    import numpy as _np
    if not eval_log:
        raise RuntimeError("No evaluations recorded in eval_log.")

    best_idx = int(_np.argmin([e["y"] for e in eval_log]))
    best_eval = eval_log[best_idx]

    # Extract the episode cogs + chi_value used in that best eval
    # (We put them into eval_log in f_obj via **episode_cogs, "chi_value": chi_value)
    best_ep_cogs = {k: v for k, v in best_eval.items()
                    if k not in {"y", "total", "respNLL", "timeMAE", "probs", "strategies", "meta_raw",
                                 "condition", "trial_type", "with_xai_used", "with_xai_requested",
                                 "mismatch_applied", "invalid_under_condition"}}
    # Pull chi_value out and remove from the cogs dict if present
    best_chi = float(best_ep_cogs.pop("chi_value"))

    return {
        "best_params": {"episode_cogs": best_ep_cogs, "chi_value": best_chi},
        "objective": float(best_eval["total"]),
        "respNLL": float(best_eval["respNLL"]),
        "timeMAE": float(best_eval["timeMAE"]),
        "probs": best_eval.get("probs"),
        "strategies": best_eval.get("strategies"),
        "best_eval_full": best_eval,      # <-- contains meta_raw -> logs, etc.
        "optimizer": "GP(skopt)",
        "n_evals": len(eval_log),
        "history": eval_log,
        "trial_type": best_eval.get("trial_type"),
    }

In [144]:
# Hardcode dataset_id for now
dataset_id = 1

# Choose what to tune (subset only)
tune_keys = ["retrieval_threshold", "latency_factor", "ddm_a", "ddm_s", "lapse", "chi_value", "T_enc"]

# Freeze anything you don’t want touched
freeze = {
    "latency_factor": 0.2,
    "lapse": 0.005,
    # "retrieval_threshold": -1.5
    # "latency_factor": 1.0,     # example
    "ddm_a": 0.0,
    # "chi_value": 0.0001, 
}

# Base episode cogs (start point + fixed keys your strategies expect)
base_episode_cogs = {
    "retrieval_threshold": -0.5,
    # "latency_factor": 0.5,
    # "ddm_a": 1.0,
    "ddm_s": 1.0,
    "T_enc": 1.0,             # if your strategies use it internally
    "T_op": 0.5,
    # "lapse": 0.05,
}

cfg = MetaFitConfig(
    w_time=0.0,                   # include timing if you have participant RTs
    time_q=0.80,                  # drop top 20% RTs as outliers
    n_calls=30,
    n_initial_points=6,
    random_state=42,
    deterministic=True,
    bounds={
        "retrieval_threshold": (-2.0, 1.5),
        "latency_factor": (0.001, 0.5),
        "ddm_a": (0.8, 1.2),
        "ddm_s": (0.4, 1.2),
        "lapse": (0.01, 0.05),
        "chi_value": (0.001, 0.005),
        "T_enc": (0.5, 3.0),
    },
    init_vals={
        "retrieval_threshold": -0.3,
        "latency_factor": 0.6,
        "ddm_a": 1.0,
        "ddm_s": 0.6,
        "lapse": 0.05,
        "chi_value": 0.001,
    }
)


In [145]:
# Get participant info
participant_ids = loader.get_participant_ids()
# Pick random participant with LR + high complexity
participant_id = random.choice([
    pid for pid in participant_ids
    if ((loader.get_participant_info(pid)['condition'] == 'LR')) &
    (loader.get_participant_info(pid)['app_id'] == 'mushrooms')# &
        # (loader.get_participant_info(pid)['complexity'] == 'high'))
])

# participant_id = "660c9fe7b2e6bf22bcc608b1"
# print(f"Optimizing for participant {participant_id}...")

participant_info = loader.get_participant_info(participant_id)

app_id     = participant_info['app_id']
model_name = participant_info['model']
condition  = participant_info['condition']
complexity = participant_info['complexity']

filtered_data_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)
# X_raw, y_raw = filtered_data_loader.load_instances([[]])

participant_instance_ids = list(loader.get_forward_trials(participant_id)["Instance Id"])
X_raw, y_raw = filtered_data_loader.load_instances(participant_instance_ids, normalize=False)
X_norm, _ = filtered_data_loader.load_instances(participant_instance_ids, normalize=True)

responses = list(loader.get_forward_trials(participant_id)["Response"])
with_xai_schedule = list(loader.get_forward_trials(participant_id)["Tested w/ XAI"])
with_xai_schedule = [int(v=="w/ XAI") for v in with_xai_schedule]
trial_type_schedule = list(loader.get_forward_trials(participant_id)["XAIType"])

# convert all to numpy arrays
X_raw = np.asarray(X_raw, dtype=np.float32)
X_norm = np.asarray(X_norm, dtype=np.float32)
y_raw = np.asarray(y_raw, dtype=np.int64)
responses = np.asarray(responses, dtype=np.int64)
with_xai_schedule = np.asarray(with_xai_schedule, dtype=np.int64)

print(participant_id, app_id, condition, complexity)


67193afeacc11c6126bded8c mushrooms LR low


In [146]:
lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id=app_id, model_name=model_name, variant="sparse" if complexity=="low" else "dense")
dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id=app_id, model_name=model_name, depth=2 if complexity=="low" else 3)

In [147]:
# 1) Load your saved *meta* model (trained on MetaRouterEnv)
meta_model = PPO.load("./models_meta/best/best_model.zip")

ddm_a_bins = 3

# lr = HeadlessLRPolicy(...)
lr_calc = HeadlessLRCalcPolicy(
    model_path="./model_calculation/simple_chi_model",   # your trained subpolicy path
    lr_exps={1: lr_exp},
    memory_factory=_make_memory,
    training_cog_params=training_cog_params,
    ddm_a_bins=ddm_a_bins,
)
# LR-heuristic headless
lr_heur = HeadlessLRHeurPolicy(
    model_path="./model_heuristic/simple_chi_model",
    lr_exps={1: lr_exp},
    memory_factory=_make_memory,
    training_cog_params=training_cog_params,
    ddm_a_bins=ddm_a_bins,
    heuristic_kwargs={"num_samples": 64, "K_top": 3},
)
# Decision Tree headless
dt = HeadlessDTPolicy(
    model_path="./model_dt/simple_chi_model.zip",      # SB3 save (zip)
    dt_exps={1: dt_exp},
    memory_factory=_make_memory,
    training_cog_params=training_cog_params,
    ddm_a_bins=ddm_a_bins,
    forbid_read_without_xai=True,
    dt_kwargs={"n_mc": 64, "topk_k": 3, "refresh_prob_cap": 1.0},
)
strategies = {"lr_calc": lr_calc, "lr_heur": lr_heur, "dt": dt}
order = ["lr_calc", "lr_heur", "dt"]  # stable order for meta action indices

In [148]:

fit_out = fit_meta_params_gp_bo(
    meta_model=meta_model,
    strategies=strategies,
    strategy_order=None,          # or a fixed list
    X_raw=X_raw,
    y_raw=y_raw,         # <-- participant RESPONSES here
    X_norm=X_norm,
    response=responses,           # participant RESPONSES here
    with_xai_schedule=with_xai_schedule,   # or None + with_xai_ratio
    with_xai_ratio=0.5,
    trial_type_schedule=trial_type_schedule,
    condition=condition,
    perm=None,
    dataset_id=1,
    training_cog_params=training_cog_params,
    tune_keys=tune_keys,
    freeze=freeze,
    base_episode_cogs=base_episode_cogs,
    cfg=cfg
)

print("Best:", fit_out["best_params"])
print("NLL:", fit_out["respNLL"], "TimeMAE:", fit_out["timeMAE"], "Obj:", fit_out["objective"])

Eval 1: total=28.4206, respNLL=28.4206, timeMAE=0.0000, cogs={'retrieval_threshold': -0.3, 'ddm_s': 0.6, 'T_enc': 1.5, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0010
Eval 2: total=46.1790, respNLL=46.1790, timeMAE=0.0000, cogs={'retrieval_threshold': 0.7879004540108157, 'ddm_s': 0.5467478318929311, 'T_enc': 1.4568327670341186, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0041
Eval 3: total=38.1172, respNLL=38.1172, timeMAE=0.0000, cogs={'retrieval_threshold': -0.4395853650124306, 'ddm_s': 0.47997993265440236, 'T_enc': 0.9091714246704565, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0028
Eval 4: total=25.3938, respNLL=25.3938, timeMAE=0.0000, cogs={'retrieval_threshold': -1.4999661372732072, 'ddm_s': 0.9207107783590823, 'T_enc': 1.8230318311770966, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0012
Eval 5: total=31.6194, respNLL=31.6194, timeMAE=0.000

In [151]:
rng = np.random.default_rng(None)  # fixed seed for reproducibility
sampled_hits = 0
resp_hits = 0
n = 0

trials_list = list(zip(
    participant_instance_ids, y_raw, with_xai_schedule,
    fit_out["trial_type"], responses, fit_out["strategies"], fit_out["probs"]
))

for (iid, y, xai, trial_type, resp, strat, probs) in trials_list:
    # normalize to 0/1 in case you have {-1, +1}
    y_bin = 1 if int(y) > 0 else 0
    resp_bin = 1 if int(resp) > 0 else 0

    # probability assigned to the participant's response (for your per-trial print)
    prob = _select_prob_of_response(probs, resp)

    # one stochastic sample from model probs, then compare to y
    p1 = float(probs[1])  # assumes probs = [p0, p1]
    sampled_label = 1 if rng.random() < p1 else 0

    sampled_hits += int(sampled_label == y_bin)
    resp_hits += int(resp_bin == y_bin)
    n += 1

    print(f"Instance Id:{int(iid)}\tWith XAI:{xai}\tXAI Shown:{trial_type}"
          f"\tResponse:{resp_bin}\tStrategy:{strat}\tProbs:{probs}"
          f"\tSelectedProb={prob:.4f}")

# ---- final summaries ----
print(f"\nFinal over {n} trials")
print(f"Sampled-from-probs match to y: {sampled_hits}/{n} = {sampled_hits/n:.2%}")
print(f"Participant response match to y: {resp_hits}/{n} = {resp_hits/n:.2%}")


Instance Id:254	With XAI:1	XAI Shown:LR	Response:0	Strategy:lr_heur	Probs:[0.6423480048635788, 0.3576519951364212]	SelectedProb=0.6423
Instance Id:192	With XAI:1	XAI Shown:LR	Response:0	Strategy:lr_calc	Probs:[0.6911930909439681, 0.30880690905603186]	SelectedProb=0.6912
Instance Id:61	With XAI:0	XAI Shown:LR	Response:1	Strategy:lr_heur	Probs:[0.40473482291043217, 0.5952651770895678]	SelectedProb=0.5953
Instance Id:374	With XAI:1	XAI Shown:LR	Response:1	Strategy:lr_calc	Probs:[0.5313562090523736, 0.4686437909476264]	SelectedProb=0.4686
Instance Id:79	With XAI:0	XAI Shown:LR	Response:1	Strategy:lr_heur	Probs:[0.3629032466344768, 0.6370967533655232]	SelectedProb=0.6371
Instance Id:23	With XAI:1	XAI Shown:LR	Response:1	Strategy:lr_calc	Probs:[0.639321362902795, 0.360678637097205]	SelectedProb=0.3607
Instance Id:390	With XAI:1	XAI Shown:LR	Response:1	Strategy:lr_calc	Probs:[0.5871934642104049, 0.4128065357895951]	SelectedProb=0.4128
Instance Id:173	With XAI:1	XAI Shown:LR	Response:0	Strateg

In [664]:
len(trials_list)

40

## Fit to several participants

In [ ]:
# Hardcode dataset_id for now
dataset_id = 1

# Choose what to tune (subset only)
tune_keys = ["retrieval_threshold", "latency_factor", "ddm_a", "ddm_s", "lapse", "chi_value", "T_enc"]

# Freeze anything you don’t want touched
freeze = {
    "latency_factor": 0.2,
    "lapse": 0.05,
    # "retrieval_threshold": -1.5
    # "latency_factor": 1.0,     # example
    "ddm_a": 0.0,
}

# Base episode cogs (start point + fixed keys your strategies expect)
base_episode_cogs = {
    "retrieval_threshold": -0.5,
    # "latency_factor": 0.5,
    # "ddm_a": 1.0,
    "ddm_s": 1.0,
    "T_enc": 2.0,             # if your strategies use it internally
    "T_op": 0.5,
    # "lapse": 0.05,
}

cfg = MetaFitConfig(
    w_time=0,                   # include timing if you have participant RTs
    time_q=0.80,                  # drop top 20% RTs as outliers
    n_calls=20,
    n_initial_points=6,
    random_state=42,
    deterministic=True,
    bounds={
        "retrieval_threshold": (-2.0, 1.5),
        "latency_factor": (0.001, 0.5),
        "ddm_a": (0.8, 1.2),
        "ddm_s": (0.2, 1.2),
        "lapse": (0.01, 0.05),
        "chi_value": (0.0, 0.02),
        "T_enc": (0.5, 3.0),
    },
    init_vals={
        "retrieval_threshold": -0.3,
        "latency_factor": 0.6,
        "ddm_a": 1.0,
        "ddm_s": 1.0,
        "lapse": 0.05,
        "chi_value": 0.001,
    }
)


In [153]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Any, Optional, Sequence
import random

# ---- helpers ----
def _sample_label_from_probs(probs: Sequence[float], rng: np.random.Generator) -> int:
    """Sample 0/1 given probs=[p0,p1]."""
    p1 = float(probs[1])
    return int(rng.random() < p1)

def _bool_to_mark(v: bool) -> str:
    return "w/ XAI" if bool(v) else "w/o XAI"

def _ensure_numpy(a, dtype=None):
    arr = np.asarray(a)
    return arr.astype(dtype) if dtype is not None else arr

# ---- main runner ----
def fit_and_dump_trials_for_participants(
    *,
    participant_ids: Sequence[str],
    loader,
    ai_dataset_loader,
    meta_model,
    training_cog_params: Dict[str, Any],
    build_lr_exp,
    build_dt_exp,
    build_strategies,
    tune_keys: Sequence[str],
    freeze: Optional[Dict[str, float]],
    base_episode_cogs: Optional[Dict[str, Any]],
    cfg,
    invalid_action_penalty: float = -1.0,
    with_xai_ratio_default: float = 0.5,
    out_path: str = "./participant_trial_dump.csv",
    rng_seed: int = 123,
):
    import numpy as np
    import pandas as pd
    from pathlib import Path

    rng = np.random.default_rng(rng_seed)
    rows = []

    for pid in participant_ids:
        info = loader.get_participant_info(pid)
        app_id     = info.get("app_id")
        model_name = info.get("model")
        condition  = info.get("condition")
        complexity = info.get("complexity")
        phase      = info.get("phase", None)

        print(f"Fitting participant {pid} | App:{app_id} Model:{model_name} Cond:{condition} Comp:{complexity}")

        # Explainers + strategies (participant-specific)
        lr_exp = build_lr_exp(app_id, model_name, complexity)
        dt_exp = build_dt_exp(app_id, model_name, complexity)
        strategies = build_strategies(lr_exp, dt_exp)

        trials_df = loader.get_forward_trials(pid)
        instance_ids = list(trials_df["Instance Id"])
        responses = np.asarray(trials_df["Response"], dtype=int)

        with_xai_schedule = None
        if "Tested w/ XAI" in trials_df.columns:
            with_xai_schedule = np.asarray(
                trials_df["Tested w/ XAI"].map(lambda v: 1 if str(v).strip().lower().startswith("w/ ") else 0),
                dtype=bool
            )
        trial_type_schedule = None
        if "XAIType" in trials_df.columns:
            trial_type_schedule = np.asarray(trials_df["XAIType"], dtype=object)

        # Features/labels for these instances
        filtered_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)
        X_raw, y_raw = filtered_loader.load_instances(instance_ids, normalize=False)
        X_norm, _    = filtered_loader.load_instances(instance_ids, normalize=True)
        X_raw  = np.asarray(X_raw, dtype=np.float32)
        X_norm = np.asarray(X_norm, dtype=np.float32)
        y_raw  = np.asarray(y_raw,  dtype=int)

        participant_times = np.asarray(trials_df["Time"], dtype=float)  # participants’ recorded RTs

        # === Fit; return best *evaluated* run (no re-run) ===
        fit_out = fit_meta_params_gp_bo(
            meta_model=meta_model,
            strategies=strategies,
            strategy_order=None,
            X_raw=X_raw,
            y_raw=y_raw,
            X_norm=X_norm,
            response=responses,
            with_xai_schedule=with_xai_schedule,
            with_xai_ratio=with_xai_ratio_default,
            trial_type_schedule=trial_type_schedule,
            condition=condition,
            perm=None,
            dataset_id=1,
            training_cog_params=training_cog_params,
            tune_keys=tune_keys,
            freeze=freeze,
            base_episode_cogs=base_episode_cogs,
            cfg=cfg,
            invalid_action_penalty=invalid_action_penalty,
        )

        best_ep   = dict(fit_out["best_params"]["episode_cogs"])
        best_chi  = float(fit_out["best_params"]["chi_value"])
        best_eval = fit_out["best_eval_full"]          # <- already includes meta_raw logs, probs, times, etc.

        # Pull logs + metrics from best_eval directly
        logs = best_eval["meta_raw"]["logs"]
        probs_list        = best_eval["probs"]               # list of [p0, p1]
        strats            = best_eval["strategies"]
        pred_times        = np.asarray(logs["pred_time"], float)          # per-trial model time
        # with_xai_req_list = logs["with_xai_"]
        xai_type_list     = logs["trial_type"]

        nll_value     = float(best_eval["respNLL"])
        objective_val = float(best_eval["total"])
        mean_time_val = float(np.mean(pred_times)) if len(pred_times) else 0.0

        # Sample model predictions from probs
        model_preds = [int(rng.random() < float(p[1])) for p in probs_list]

        # Which cognitive params to repeat
        cog_cols = sorted(set(best_ep.keys()) | {"chi_value"})

        def mark_xai(b): return "w/ XAI" if bool(b) else "w/o XAI"

        for t, iid in enumerate(instance_ids):
            response_equals_ai = int(responses[t]>0) if int(y_raw[t])==1 else int(responses[t]<0)
            model_equals_ai = int(model_preds[t])==int(y_raw[t])

            row = {
                "Participant Id": pid,
                "Condition": condition,
                "Model": model_name,
                "AppId": app_id,
                "Complexity": complexity,
                "Phase": phase,
                "Trial Index": t,
                "Instance Id": iid,
                "XAIType": (xai_type_list[t] if xai_type_list is not None else None),
                "Tested w/ XAI": with_xai_schedule[t] if with_xai_schedule is not None else None,
                # === Per-trial model time (explicit) ===
                "Model Pred Time": float(pred_times[t]),
                "Time": float(participant_times[t]),  # keep your original column name if downstream expects it
                "Response": int(responses[t]),
                "AI Prediction (y)": int(y_raw[t]),
                "Model Prediction": int(model_preds[t]),
                "Response==AI": int(response_equals_ai),
                "Model==AI": int(model_equals_ai),
                # Repeat participant-level metrics per row
                "NLL": nll_value,
                "Mean Pred Time": mean_time_val,
                "Objective": objective_val,
                # Optional debug
                "Strategy": str(strats[t]) if t < len(strats) else None,
            }
            for k in cog_cols:
                row[k] = best_chi if k == "chi_value" else best_ep.get(k, None)
            rows.append(row)

    df = pd.DataFrame(rows)

    fixed_cols = [
        "Participant Id","Condition","Model","AppId","Complexity","Phase",
        "Trial Index","Instance Id","XAIType","Tested w/ XAI",
        # put the explicit model-time columns up front
        "Model Pred Time","Time",
        "Response","AI Prediction (y)","Model Prediction",
        "NLL","Mean Pred Time","Objective",
    ]
    remaining = [c for c in df.columns if c not in fixed_cols]
    df = df[fixed_cols + remaining]

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return out_path, df



In [154]:
# Strategy builders (use your classes & per-participant explainers)
def build_lr_exp(app_id, model_name, complexity):
    return LogisticRegressionInterpreter(
        lr_df, metadata_df, app_id=app_id, model_name=model_name,
        variant=("sparse" if complexity=="low" else "dense")
    )

def build_dt_exp(app_id, model_name, complexity):
    return DecisionTreeInterpreter(
        dt_df, metadata_df, app_id=app_id, model_name=model_name,
        depth=(2 if complexity=="low" else 3)
    )

def build_strategies(lr_exp, dt_exp):
    ddm_a_bins = 3
    lr_calc = HeadlessLRCalcPolicy(
        model_path="./model_calculation/simple_chi_model",
        lr_exps={1: lr_exp},
        memory_factory=_make_memory,
        training_cog_params=training_cog_params,
        ddm_a_bins=ddm_a_bins,
    )
    lr_heur = HeadlessLRHeurPolicy(
        model_path="./model_heuristic/simple_chi_model",
        lr_exps={1: lr_exp},
        memory_factory=_make_memory,
        training_cog_params=training_cog_params,
        ddm_a_bins=ddm_a_bins,
        heuristic_kwargs={"num_samples": 64, "K_top": 3},
    )
    dt = HeadlessDTPolicy(
        model_path="./model_dt/simple_chi_model.zip",
        dt_exps={1: dt_exp},
        memory_factory=_make_memory,
        training_cog_params=training_cog_params,
        ddm_a_bins=ddm_a_bins,
        forbid_read_without_xai=True,
        dt_kwargs={"n_mc": 64, "topk_k": 3, "refresh_prob_cap": 1.0},
    )
    return {"lr_calc": lr_calc, "lr_heur": lr_heur, "dt": dt}

# Choose the participants you want
participant_ids = loader.get_participant_ids()
some_participants =[
    pid for pid in participant_ids
    if (loader.get_participant_info(pid)['app_id'] == 'mushrooms')# &
        # (loader.get_participant_info(pid)['complexity'] == 'high'))
]


out_path, df = fit_and_dump_trials_for_participants(
    participant_ids=some_participants,
    loader=loader,
    ai_dataset_loader=ai_dataset_loader,
    meta_model=PPO.load("./models_meta/best/best_model.zip"),
    training_cog_params=training_cog_params,
    build_lr_exp=build_lr_exp,
    build_dt_exp=build_dt_exp,
    build_strategies=build_strategies,
    tune_keys=tune_keys,
    freeze=freeze,
    base_episode_cogs=base_episode_cogs,
    cfg=cfg,
    invalid_action_penalty=-1.0,
    out_path="./exports/participant_trial_dump.csv",
)

print("Saved to:", out_path)
display(df.head())


Fitting participant 65006385ae3b0aa61a642e29 | App:mushrooms Model:mlp Cond:DT Comp:high
Eval 1: total=38.2298, respNLL=38.2298, timeMAE=0.0000, cogs={'retrieval_threshold': -0.3, 'ddm_s': 1.0, 'T_enc': 1.5, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0010
Eval 2: total=27.5672, respNLL=27.5672, timeMAE=0.0000, cogs={'retrieval_threshold': 0.7879004540108157, 'ddm_s': 0.38343478986616386, 'T_enc': 1.4568327670341186, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0156
Eval 3: total=36.1649, respNLL=36.1649, timeMAE=0.0000, cogs={'retrieval_threshold': -0.4395853650124306, 'ddm_s': 0.29997491581800295, 'T_enc': 0.9091714246704565, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.0, 'lapse': 0.005}, chi_value=0.0092
Eval 4: total=47.2475, respNLL=47.2475, timeMAE=0.0000, cogs={'retrieval_threshold': -1.4999661372732072, 'ddm_s': 0.8508884729488531, 'T_enc': 1.8230318311770966, 'T_op': 0.5, 'latency_factor': 0.2, 'ddm_a': 0.

,Participant Id,Condition,Model,AppId,Complexity,Phase,Trial Index,Instance Id,XAIType,Tested w/ XAI,...,T_enc,T_op,chi_value,ddm_a,ddm_s,lapse,latency_factor,p_resp,pred_time,retrieval_threshold
0,65006385ae3b0aa61a642e29,DT,mlp,mushrooms,high,None,0,197.0,DT,True,...,1.526451,0.5,0.02,0.0,0.451753,0.005,0.2,"[0.5, 0.5, 0.5388671875, 0.5, 0.5, 0.5, 0.5, 0...","[0.1207046341129352, 0.12070456139910041, 5.01...",1.010615
1,65006385ae3b0aa61a642e29,DT,mlp,mushrooms,high,None,1,320.0,DT,False,...,1.526451,0.5,0.02,0.0,0.451753,0.005,0.2,"[0.5, 0.5, 0.5388671875, 0.5, 0.5, 0.5, 0.5, 0...","[0.1207046341129352, 0.12070456139910041, 5.01...",1.010615
2,65006385ae3b0aa61a642e29,DT,mlp,mushrooms,high,None,2,323.0,DT,False,...,1.526451,0.5,0.02,0.0,0.451753,0.005,0.2,"[0.5, 0.5, 0.5388671875, 0.5, 0.5, 0.5, 0.5, 0...","[0.1207046341129352, 0.12070456139910041, 5.01...",1.010615
3,65006385ae3b0aa61a642e29,DT,mlp,mushrooms,high,None,3,383.0,DT,False,...,1.526451,0.5,0.02,0.0,0.451753,0.005,0.2,"[0.5, 0.5, 0.5388671875, 0.5, 0.5, 0.5, 0.5, 0...","[0.1207046341129352, 0.12070456139910041, 5.01...",1.010615
4,65006385ae3b0aa61a642e29,DT,mlp,mushrooms,high,None,4,376.0,DT,False,...,1.526451,0.5,0.02,0.0,0.451753,0.005,0.2,"[0.5, 0.5, 0.5388671875, 0.5, 0.5, 0.5, 0.5, 0...","[0.1207046341129352, 0.12070456139910041, 5.01...",1.010615


## Generate data using the parameters

In [668]:
def run_with_shared_params_and_dump(
    *,
    participant_ids: Sequence[str],
    loader,
    ai_dataset_loader,
    meta_model,
    training_cog_params: Dict[str, Any],
    build_lr_exp,
    build_dt_exp,
    build_strategies,
    # --- shared params for ALL participants ---
    episode_cogs: Dict[str, Any],
    chi_value: float,
    deterministic: bool = True,
    with_xai_ratio: float = 0.5,
    # ---
    invalid_action_penalty: float = -1.0,
    out_path: str = "./participant_trial_dump.csv",
    rng_seed: int = 123,
):
    """
    Runs ONE meta-episode per participant using the SAME (pre-specified) parameters for all,
    then writes a per-trial CSV. No NLL/cognitive-parameter columns are included.

    CSV columns:
      Participant Id, Condition, Model, AppId, Complexity, Phase,
      Trial Index, Instance Id, XAIType, Tested w/ XAI,
      Model Pred Time, Time,
      Response, AI Prediction (y), Model Prediction,
      Objective
    """
    import numpy as np
    import pandas as pd
    from pathlib import Path

    rng = np.random.default_rng(rng_seed)
    rows = []

    for pid in participant_ids:
        print(f"Running participant {pid}...")
        # --- participant info
        info = loader.get_participant_info(pid)
        app_id     = info.get("app_id")
        model_name = info.get("model")
        condition  = info.get("condition")
        complexity = info.get("complexity")
        phase      = info.get("phase", None)

        # --- explainers + strategies
        lr_exp = build_lr_exp(app_id, model_name, complexity)
        dt_exp = build_dt_exp(app_id, model_name, complexity)
        strategies = build_strategies(lr_exp, dt_exp)

        # --- trials & schedules
        trials_df = loader.get_forward_trials(pid)
        instance_ids = list(trials_df["Instance Id"])
        responses    = np.asarray(trials_df["Response"], dtype=int)

        with_xai_schedule = None
        if "Tested w/ XAI" in trials_df.columns:
            with_xai_schedule = np.asarray(
                trials_df["Tested w/ XAI"].map(lambda v: 1 if str(v).strip().lower().startswith("w/ ") else 0),
                dtype=bool
            )
        trial_type_schedule = None
        if "XAIType" in trials_df.columns:
            trial_type_schedule = np.asarray(trials_df["XAIType"], dtype=object)

        # --- features/labels
        fl = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)
        X_raw, y_raw = fl.load_instances(instance_ids, normalize=False)
        X_norm, _    = fl.load_instances(instance_ids, normalize=True)
        X_raw  = np.asarray(X_raw, dtype=np.float32)
        X_norm = np.asarray(X_norm, dtype=np.float32)
        y_raw  = np.asarray(y_raw,  dtype=int)

        participant_times = np.asarray(trials_df["Time"], dtype=float) if "Time" in trials_df.columns else np.zeros(len(instance_ids), float)

        # --- run one meta-episode (shared params)
        meta_out = meta_episode_likelihood(
            meta_model=meta_model,
            strategies=strategies,
            strategy_order=None,
            X_raw=X_raw,
            y_raw=y_raw,                 # not used inside for NLL; meta uses responses
            X_norm=X_norm,
            response=responses,
            with_xai_schedule=with_xai_schedule,
            with_xai_ratio=with_xai_ratio,
            trial_type_schedule=trial_type_schedule,
            condition=condition,
            perm=None,
            dataset_id=1,
            episode_cogs=episode_cogs,               # <--- SAME for all
            training_cog_params=training_cog_params,
            chi_value=float(chi_value),              # <--- SAME for all
            deterministic=bool(deterministic),
        )

        logs        = meta_out["meta_raw"]["logs"]
        probs_list  = meta_out["probs"]                      # list of [p0, p1]
        pred_times  = np.asarray(logs["pred_time"], float)
        xai_types   = logs.get("trial_type", None)
        objective   = float(meta_out.get("total", np.nan))   # may include timing if configured

        # sample concrete predictions from probs (optional but handy)
        model_preds = [int(rng.random() < float(p[1])) for p in probs_list]

        # dump rows
        for t, iid in enumerate(instance_ids):
            rows.append({
                "Participant Id": pid,
                "Condition": condition,
                "Model": model_name,
                "AppId": app_id,
                "Complexity": complexity,
                "Phase": phase,
                "Trial Index": t,
                "Instance Id": iid,
                "XAIType": (xai_types[t] if xai_types is not None and t < len(xai_types) else None),
                "Tested w/ XAI": (with_xai_schedule[t] if with_xai_schedule is not None else None),
                "Model Pred Time": float(pred_times[t]) if t < len(pred_times) else np.nan,
                "Time": float(participant_times[t]) if t < len(participant_times) else np.nan,
                "Response": int(responses[t]),
                "AI Prediction (y)": int(y_raw[t]),
                "Model Prediction": int(model_preds[t]),
                "Objective": objective,
            })

    # --- build & save CSV
    df = pd.DataFrame(rows)
    fixed_cols = [
        "Participant Id","Condition","Model","AppId","Complexity","Phase",
        "Trial Index","Instance Id","XAIType","Tested w/ XAI",
        "Model Pred Time","Time",
        "Response","AI Prediction (y)","Model Prediction",
        "Objective",
    ]
    remaining = [c for c in df.columns if c not in fixed_cols]
    df = df[fixed_cols + remaining]

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    return out_path, df

In [669]:
shared_episode_cogs = {"T_enc":1.5, "T_op":0.3, "latency_factor":0.2, "ddm_a":1.2, "ddm_s":0.9, "lapse":0.04, "retrieval_threshold":0.0}
out_path, df = run_with_shared_params_and_dump(
    participant_ids=some_participants,
    loader=loader,
    ai_dataset_loader=ai_dataset_loader,
    meta_model=meta_model,
    training_cog_params=training_cog_params,
    build_lr_exp=build_lr_exp,
    build_dt_exp=build_dt_exp,
    build_strategies=build_strategies,
    episode_cogs=shared_episode_cogs,
    chi_value=0.01,
    deterministic=True,
    with_xai_ratio=0.5,
    out_path="./shared_params_trial_dump.csv",
)

Running participant 65006385ae3b0aa61a642e29...
Running participant 67acfb244f95ff16b8c04704...
Running participant 67abadae1c6d8443b84ce687...
Running participant 670c2cc6d47a5af30e54a1b7...
Running participant 611202ff20bc8f6143b2b906...
Running participant 6144b21806fb764e5f179bc2...
Running participant 67413944bea8f72bf0e85ee0...
Running participant 68111c116d884447584ec884...
Running participant 66707f3a4435f4e05d0940a1...
Running participant 67d1f22e4a4e2154f7069121...
Running participant 66fd4cf0290e991fb59105ee...
Running participant 6646555bb6bf80457ea8dac4...
Running participant 660c9fe7b2e6bf22bcc608b1...
Running participant 677cc413f40496079c7a821e...
Running participant 66fae4bae71ae61cda501834...
Running participant 67217069ec642ab038feec41...
Running participant 66463a2c0a36377060b1910c...
Running participant 66ad139637e066318f759155...
Running participant 6789809f8a3360391e622469...
Running participant 67446a0309a2bfd11c4ac73d...
Running participant 61301afc08a5e9d282fc